GPT-2 Model Configuration

In [29]:
GPT_CONFIG_124M = {
    "vocab_size": 50527,
    "context_length": 256,
    "stride": 128,
    "embedding_dim": 768,
    "num_layers": 12,
    "num_heads": 12,
    "dropout": 0.1,
    "qkv_bias": False
}

Load Dataset

In [22]:
file_path = "../romeo-juliet.txt"

with open(file_path, 'r', encoding = 'utf-8') as file:
    text_data = file.read()

In [23]:
print(text_data)

The Project Gutenberg eBook of Romeo and Juliet
    
This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this ebook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: Romeo and Juliet

Author: William Shakespeare

Release date: November 1, 1998 [eBook #1513]
                Most recently updated: September 18, 2025

Language: English

Credits: the PG Shakespeare Team, a team of about twenty Project Gutenberg volunteers


*** START OF THE PROJECT GUTENBERG EBOOK ROMEO AND JULIET ***




THE TRAGEDY OF ROMEO AND JULIET

by William Shakespeare




Contents

THE PROLOGUE.

ACT I
Scene I. A public place.
Scene II. A Street.
Scene III. Room in Capulet’s House.


Tokenization

In [37]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [25]:
token_ids = tokenizer.encode(text_data)
print(token_ids)

[464, 4935, 20336, 46566, 286, 43989, 290, 38201, 198, 220, 220, 220, 220, 198, 1212, 47179, 318, 329, 262, 779, 286, 2687, 6609, 287, 262, 1578, 1829, 290, 198, 1712, 584, 3354, 286, 262, 995, 379, 645, 1575, 290, 351, 2048, 645, 8733, 198, 10919, 15485, 13, 921, 743, 4866, 340, 11, 1577, 340, 1497, 393, 302, 12, 1904, 340, 739, 262, 2846, 198, 1659, 262, 4935, 20336, 13789, 3017, 351, 428, 47179, 393, 2691, 198, 265, 7324, 13, 70, 19028, 13, 2398, 13, 1002, 345, 389, 407, 5140, 287, 262, 1578, 1829, 11, 198, 5832, 481, 423, 284, 2198, 262, 3657, 286, 262, 1499, 810, 345, 389, 5140, 198, 19052, 1262, 428, 46566, 13, 198, 198, 19160, 25, 43989, 290, 38201, 198, 198, 13838, 25, 3977, 22197, 198, 198, 26362, 3128, 25, 3389, 352, 11, 7795, 685, 68, 10482, 1303, 1314, 1485, 60, 198, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 4042, 2904, 6153, 25, 2693, 1248, 11, 32190, 198, 198, 32065, 25, 3594, 198, 198, 42855, 25, 262, 23842, 22197, 4816, 11, 257, 1074, 28

In [26]:
len(token_ids)

50325

Implementing Dataloader

In [49]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, text, tokenizer, max_len, stride):

        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text)

        for i in range(0, len(token_ids) - max_len, stride):
            input_chunk = token_ids[i:i+max_len]
            target_chunk = token_ids[i+1:i+max_len+1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]
    
def create_dataloader(text, tokenizer, batch_size, context_len, stride, shuffle, drop_last, num_workers):

    dataset = GPTDataset(text, tokenizer, context_len, stride)

    dataloader = DataLoader(

        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers    
    )

    return dataloader

In [47]:
train_ratio = 0.8

train_data = text_data[:int(train_ratio*len(text_data))]
test_data = text_data[int(train_ratio*len(text_data)):]

train_dataloader = create_dataloader(

    text=train_data,
    tokenizer=tokenizer,
    batch_size=128,
    context_len=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["stride"],
    shuffle=True,
    drop_last=True,
    num_workers=0

)

test_dataloader = create_dataloader(
    
    text=test_data,
    tokenizer=tokenizer,
    batch_size=64,
    context_len=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["stride"],
    shuffle=False,
    drop_last=False,
    num_workers=0

)

In [48]:
for x, y in train_dataloader:
    print(x.shape, y.shape)

for x, y in test_dataloader:
    print(x.shape, y.shape)


torch.Size([128, 256]) torch.Size([128, 256])
torch.Size([128, 256]) torch.Size([128, 256])
torch.Size([64, 256]) torch.Size([64, 256])
torch.Size([2, 256]) torch.Size([2, 256])


Multi-Head Attention

In [ ]:
class MultiHeadAttention()